In [ ]:
# CIFAR-10 Data Set : Train the model (including the steps Batch Normalization + DropOut)

In [ ]:
!pip install torch torchvision matplotlib numpy

=============================================================================
  BATCH NORMALIZATION (all types) + DROPOUT — Complete Guide with PyTorch
=============================================================================
  Topics:
    1. Why Normalization?  (Internal Covariate Shift)
    2. Batch Normalization  (BatchNorm1d, BatchNorm2d, BatchNorm3d)
    3. Layer Normalization
    4. Instance Normalization
    5. Group Normalization
    6. Spectral Normalization
    7. Dropout (standard)
    8. Dropout2d / Dropout3d
    9. Alpha Dropout
   10. How to SELECT the right dropout rate (empirical + theoretical)
   11. Full experiments on CIFAR-10 comparing all combinations
   12. Plots: accuracy curves, feature distributions, ablation study

  Dataset : CIFAR-10 (auto-downloaded via torchvision)
  Requires: pip install torch torchvision matplotlib numpy
=============================================================================

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS      = 8
BATCH_SIZE  = 128
LR          = 1e-3
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR    = "./data"
OUT_DIR     = "./bn_dropout_results"
os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 72)
print("  BATCH NORMALIZATION & DROPOUT — Comprehensive Guide")
print("=" * 72)
print(f"  Device : {DEVICE}   |   Epochs : {EPOCHS}   |   Batch : {BATCH_SIZE}")
print("=" * 72)


  BATCH NORMALIZATION & DROPOUT — Comprehensive Guide
  Device : cuda   |   Epochs : 8   |   Batch : 128


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATASET  (CIFAR-10)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1] Loading CIFAR-10 …")
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
])
train_full = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=train_tf)
test_full  = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)

# Use a subset for speed (first 10 000 train, 2 000 test)
train_ds = Subset(train_full, range(10_000))
test_ds  = Subset(test_full,  range(2_000))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

CLASSES = train_full.classes
print(f"  Train: {len(train_ds):,}  |  Test: {len(test_ds):,}  |  Classes: {CLASSES}")



[1] Loading CIFAR-10 …


100%|██████████| 170M/170M [00:04<00:00, 42.4MB/s]


  Train: 10,000  |  Test: 2,000  |  Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  NORMALIZATION LAYER DEMONSTRATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("  [3] NORMALIZATION LAYER SHAPES — Minimal Examples")
print("="*72)

# Fake batch: 4 samples, 8 channels, 6×6 spatial
x = torch.randn(4, 8, 6, 6)
print(f"\n  Input tensor shape: {list(x.shape)}  (N=4, C=8, H=6, W=6)\n")

norm_layers = [
    ("BatchNorm2d",    nn.BatchNorm2d(8)),
    ("LayerNorm",      nn.LayerNorm([8, 6, 6])),
    ("InstanceNorm2d", nn.InstanceNorm2d(8, affine=True)),
    ("GroupNorm(G=2)", nn.GroupNorm(num_groups=2, num_channels=8)),
    ("GroupNorm(G=8)", nn.GroupNorm(num_groups=8, num_channels=8)),
]

for name, layer in norm_layers:
    y = layer(x)
    print(f"  {name:<22} → output: {list(y.shape)}  "
          f"| trainable params: {sum(p.numel() for p in layer.parameters())}")

# 1D / 3D variants
x1 = torch.randn(4, 16)       # (N, C)
x3 = torch.randn(2, 4, 8, 8, 8)  # (N, C, D, H, W)
print(f"\n  BatchNorm1d(16) on shape {list(x1.shape)} → {list(nn.BatchNorm1d(16)(x1).shape)}")
print(f"  BatchNorm3d(4)  on shape {list(x3.shape)} → {list(nn.BatchNorm3d(4)(x3).shape)}")



  [3] NORMALIZATION LAYER SHAPES — Minimal Examples

  Input tensor shape: [4, 8, 6, 6]  (N=4, C=8, H=6, W=6)

  BatchNorm2d            → output: [4, 8, 6, 6]  | trainable params: 16
  LayerNorm              → output: [4, 8, 6, 6]  | trainable params: 576
  InstanceNorm2d         → output: [4, 8, 6, 6]  | trainable params: 16
  GroupNorm(G=2)         → output: [4, 8, 6, 6]  | trainable params: 16
  GroupNorm(G=8)         → output: [4, 8, 6, 6]  | trainable params: 16

  BatchNorm1d(16) on shape [4, 16] → [4, 16]
  BatchNorm3d(4)  on shape [2, 4, 8, 8, 8] → [2, 4, 8, 8, 8]


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  NORMALIZATION LAYER DEMONSTRATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("  [3] NORMALIZATION LAYER SHAPES — Minimal Examples")
print("="*72)

# Fake batch: 4 samples, 8 channels, 6×6 spatial
x = torch.randn(4, 8, 6, 6)
print(f"\n  Input tensor shape: {list(x.shape)}  (N=4, C=8, H=6, W=6)\n")

norm_layers = [
    ("BatchNorm2d",    nn.BatchNorm2d(8)),
    ("LayerNorm",      nn.LayerNorm([8, 6, 6])),
    ("InstanceNorm2d", nn.InstanceNorm2d(8, affine=True)),
    ("GroupNorm(G=2)", nn.GroupNorm(num_groups=2, num_channels=8)),
    ("GroupNorm(G=8)", nn.GroupNorm(num_groups=8, num_channels=8)),
]

for name, layer in norm_layers:
    y = layer(x)
    print(f"  {name:<22} → output: {list(y.shape)}  "
          f"| trainable params: {sum(p.numel() for p in layer.parameters())}")

# 1D / 3D variants
x1 = torch.randn(4, 16)       # (N, C)
x3 = torch.randn(2, 4, 8, 8, 8)  # (N, C, D, H, W)
print(f"\n  BatchNorm1d(16) on shape {list(x1.shape)} → {list(nn.BatchNorm1d(16)(x1).shape)}")
print(f"  BatchNorm3d(4)  on shape {list(x3.shape)} → {list(nn.BatchNorm3d(4)(x3).shape)}")



  [3] NORMALIZATION LAYER SHAPES — Minimal Examples

  Input tensor shape: [4, 8, 6, 6]  (N=4, C=8, H=6, W=6)

  BatchNorm2d            → output: [4, 8, 6, 6]  | trainable params: 16
  LayerNorm              → output: [4, 8, 6, 6]  | trainable params: 576
  InstanceNorm2d         → output: [4, 8, 6, 6]  | trainable params: 16
  GroupNorm(G=2)         → output: [4, 8, 6, 6]  | trainable params: 16
  GroupNorm(G=8)         → output: [4, 8, 6, 6]  | trainable params: 16

  BatchNorm1d(16) on shape [4, 16] → [4, 16]
  BatchNorm3d(4)  on shape [2, 4, 8, 8, 8] → [2, 4, 8, 8, 8]


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  MODEL DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    """Conv → Norm → ReLU with pluggable normalization."""
    def __init__(self, in_c, out_c, norm_type="batch", num_groups=8):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 3, padding=1, bias=(norm_type == "none"))
        if norm_type == "batch":
            self.norm = nn.BatchNorm2d(out_c)
        elif norm_type == "layer":
            self.norm = None   # applied after pooling in forward
        elif norm_type == "instance":
            self.norm = nn.InstanceNorm2d(out_c, affine=True)
        elif norm_type == "group":
            g = min(num_groups, out_c)
            while out_c % g != 0: g -= 1
            self.norm = nn.GroupNorm(g, out_c)
        else:
            self.norm = None
        self.norm_type = norm_type

    def forward(self, x):
        x = self.conv(x)
        if self.norm is not None:
            x = self.norm(x)
        return F.relu(x)


class SmallCNN(nn.Module):
    """Small CNN for CIFAR-10, parameterised by norm type & dropout rate."""
    def __init__(self, norm_type="batch", dropout_rate=0.3):
        super().__init__()
        self.norm_type    = norm_type
        self.dropout_rate = dropout_rate

        self.features = nn.Sequential(
            ConvBlock(3,  32, norm_type),
            ConvBlock(32, 32, norm_type),
            nn.MaxPool2d(2),                                 # 16×16
            nn.Dropout2d(dropout_rate * 0.5),
            ConvBlock(32, 64, norm_type),
            ConvBlock(64, 64, norm_type),
            nn.MaxPool2d(2),                                 # 8×8
            nn.Dropout2d(dropout_rate * 0.5),
            ConvBlock(64, 128, norm_type),
            nn.MaxPool2d(2),                                 # 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.BatchNorm1d(256) if norm_type == "batch" else nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class NoBN_NoDropout(nn.Module):
    """Vanilla CNN — no normalization, no dropout (baseline)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Linear(256, 128),     nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, device=DEVICE):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss = tot_correct = tot = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            loss = F.cross_entropy(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot_loss    += loss.item() * X.size(0)
            tot_correct += (out.argmax(1) == y).sum().item()
            tot         += X.size(0)
    return tot_loss / tot, tot_correct / tot


def train_model(model, epochs=EPOCHS, lr=LR, tag=""):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
    t0 = time.time()
    for ep in range(1, epochs+1):
        tl, ta = run_epoch(model, train_loader, opt)
        vl, va = run_epoch(model, test_loader)
        sched.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta);  history["val_acc"].append(va)
        gap = ta - va
        print(f"    {tag}  Ep {ep:2d}/{epochs} | "
              f"Train {ta*100:.1f}% loss {tl:.3f} | "
              f"Val {va*100:.1f}% loss {vl:.3f} | gap {gap*100:.1f}%")
    history["time"] = time.time() - t0
    history["final_val_acc"]  = history["val_acc"][-1]
    history["final_train_acc"] = history["train_acc"][-1]
    history["overfit_gap"] = history["final_train_acc"] - history["final_val_acc"]
    return history


# ─────────────────────────────────────────────────────────────────────────────
# 7.  EXPERIMENT A — Compare Normalization Types
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("  [5] EXPERIMENT A: Normalization Types (dropout fixed at 0.3)")
print("="*72)

NORM_CONFIGS = [
    ("No Norm + No Dropout", lambda: NoBN_NoDropout(),                   "#95a5a6"),
    ("Batch Norm",           lambda: SmallCNN("batch",    0.3),          "#e74c3c"),
    ("Layer Norm",           lambda: SmallCNN("layer",    0.3),          "#3498db"),
    ("Instance Norm",        lambda: SmallCNN("instance", 0.3),          "#2ecc71"),
    ("Group Norm",           lambda: SmallCNN("group",    0.3),          "#f39c12"),
]

norm_results = {}
for name, build_fn, color in NORM_CONFIGS:
    print(f"\n  ── {name} ──")
    model = build_fn().to(DEVICE)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"     Params: {params:,}")
    norm_results[name] = train_model(model, tag=name[:12])
    norm_results[name]["color"] = color


# ─────────────────────────────────────────────────────────────────────────────
# 8.  EXPERIMENT B — Dropout Rate Ablation
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("  [6] EXPERIMENT B: Dropout Rate Ablation (BatchNorm fixed)")
print("="*72)
print("""
  We train the SAME architecture (BatchNorm) with 6 dropout values.
  Goal: find the rate that gives the best VAL accuracy & smallest
        train/val gap (generalisation proxy).
""")

DROP_RATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
DROP_COLORS = ["#e74c3c","#e67e22","#f1c40f","#2ecc71","#3498db","#9b59b6"]

drop_results = {}
for p, col in zip(DROP_RATES, DROP_COLORS):
    name = f"p={p:.1f}"
    print(f"\n  ── Dropout {name} ──")
    model = SmallCNN("batch", dropout_rate=p).to(DEVICE)
    drop_results[name] = train_model(model, tag=name)
    drop_results[name]["color"] = col
    drop_results[name]["p"]     = p


# ─────────────────────────────────────────────────────────────────────────────
# 9.  RESULTS SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("  NORMALIZATION RESULTS SUMMARY")
print("="*72)
print(f"  {'Config':<28} {'Val Acc':>8} {'Train Acc':>10} {'Overfit Gap':>12}")
print(f"  {'─'*28} {'─'*8} {'─'*10} {'─'*12}")
for name, r in sorted(norm_results.items(), key=lambda x: -x[1]["final_val_acc"]):
    print(f"  {name:<28} {r['final_val_acc']*100:>7.2f}%  "
          f"{r['final_train_acc']*100:>9.2f}%  "
          f"{r['overfit_gap']*100:>11.2f}%")

print("\n" + "="*72)
print("  DROPOUT ABLATION RESULTS SUMMARY")
print("="*72)
print(f"  {'Dropout p':>10} {'Val Acc':>10} {'Train Acc':>10} {'Gap':>8} {'Recommendation'}")
print(f"  {'─'*10} {'─'*10} {'─'*10} {'─'*8} {'─'*20}")
best_p = max(drop_results.items(), key=lambda x: x[1]["final_val_acc"])[0]
for name, r in drop_results.items():
    tag = " ◀ BEST" if name == best_p else ""
    gap = r["overfit_gap"] * 100
    if r["p"] == 0.0:
        rec = "Overfit risk"
    elif r["p"] <= 0.2:
        rec = "Low regularisation"
    elif r["p"] <= 0.4:
        rec = "Balanced ✓"
    else:
        rec = "High — may underfit"
    print(f"  {name:>10}  {r['final_val_acc']*100:>9.2f}%  "
          f"{r['final_train_acc']*100:>9.2f}%  {gap:>7.2f}%  {rec}{tag}")


# ─────────────────────────────────────────────────────────────────────────────
# 10.  PLOTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n  [7] Generating plots …")
EPS = list(range(1, EPOCHS + 1))

# ── Plot 1: Norm type comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Experiment A — Normalization Types on CIFAR-10", fontsize=14, fontweight="bold")

for name, r in norm_results.items():
    c = r["color"]
    axes[0].plot(EPS, [v*100 for v in r["val_acc"]], "o-", color=c,
                 linewidth=2, label=name, markersize=5)
    axes[1].plot(EPS, r["val_loss"], "o-", color=c,
                 linewidth=2, label=name, markersize=5)

axes[0].set_title("Validation Accuracy (%)"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
p1 = os.path.join(OUT_DIR, "01_norm_comparison.png")
plt.savefig(p1, dpi=120, bbox_inches="tight"); plt.close()
print(f"    Saved: {p1}")

# ── Plot 2: Train vs Val gap per norm ─────────────────────────────────────────
fig, axes = plt.subplots(1, len(norm_results), figsize=(20, 4), sharey=False)
fig.suptitle("Train vs Val Accuracy — Overfitting Diagnosis by Norm Type",
             fontsize=13, fontweight="bold")

for ax, (name, r) in zip(axes, norm_results.items()):
    ax.plot(EPS, [v*100 for v in r["train_acc"]], "o-", color=r["color"],
            linewidth=2, label="Train", markersize=4)
    ax.plot(EPS, [v*100 for v in r["val_acc"]], "s--", color=r["color"],
            linewidth=1.5, label="Val", alpha=0.7, markersize=4)
    ax.fill_between(EPS, [v*100 for v in r["train_acc"]],
                    [v*100 for v in r["val_acc"]], alpha=0.1, color=r["color"],
                    label=f"Gap≈{r['overfit_gap']*100:.1f}%")
    ax.set_title(name, fontsize=9, fontweight="bold", color=r["color"])
    ax.set_xlabel("Epoch", fontsize=8); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
p2 = os.path.join(OUT_DIR, "02_overfit_diagnosis.png")
plt.savefig(p2, dpi=120, bbox_inches="tight"); plt.close()
print(f"    Saved: {p2}")

# ── Plot 3: Dropout ablation ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Experiment B — Dropout Rate Ablation on CIFAR-10 (BatchNorm)",
             fontsize=13, fontweight="bold")

for name, r in drop_results.items():
    c = r["color"]
    axes[0].plot(EPS, [v*100 for v in r["val_acc"]], "o-", color=c,
                 linewidth=2, label=name, markersize=5)
    axes[1].plot(EPS, [v*100 for v in r["train_acc"]], "o-", color=c,
                 linewidth=2, label=name, markersize=5)

axes[0].set_title("Validation Accuracy"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Training Accuracy"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Bar: final val acc vs p
ps   = [r["p"]                  for r in drop_results.values()]
accs = [r["final_val_acc"]*100  for r in drop_results.values()]
gaps = [r["overfit_gap"]*100    for r in drop_results.values()]
cols = [r["color"]              for r in drop_results.values()]

x_pos = np.arange(len(ps))
bars = axes[2].bar(x_pos, accs, color=cols, edgecolor="white", linewidth=0.7, zorder=3)
axes[2].bar_label(bars, fmt="%.1f%%", padding=2, fontsize=8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f"p={p}" for p in ps])
axes[2].set_title("Final Val Acc vs Dropout Rate")
axes[2].set_ylabel("Validation Accuracy (%)")
axes[2].set_ylim(min(accs)-5, max(accs)+5)
axes[2].grid(True, axis="y", alpha=0.3, zorder=0)

# Mark best
best_idx = np.argmax(accs)
axes[2].get_children()[best_idx].set_edgecolor("gold")
axes[2].get_children()[best_idx].set_linewidth(2.5)

plt.tight_layout()
p3 = os.path.join(OUT_DIR, "03_dropout_ablation.png")
plt.savefig(p3, dpi=120, bbox_inches="tight"); plt.close()
print(f"    Saved: {p3}")

# ── Plot 4: Gap vs dropout rate (overfit indicator) ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Dropout Rate Selection Guide — Gap Analysis", fontsize=13, fontweight="bold")

ax1.plot(ps, gaps, "o-", color="#e74c3c", linewidth=2.5, markersize=8)
ax1.axhline(0, color="gray", linestyle="--", linewidth=1)
ax1.fill_between(ps, gaps, 0, where=[g > 5 for g in gaps], alpha=0.15,
                 color="red", label="Overfitting zone")
ax1.fill_between(ps, gaps, 0, where=[g <= 5 for g in gaps], alpha=0.15,
                 color="green", label="Healthy zone")
ax1.set_xlabel("Dropout Rate p"); ax1.set_ylabel("Train−Val Accuracy Gap (%)")
ax1.set_title("Overfitting Gap vs Dropout")
ax1.legend(); ax1.grid(True, alpha=0.3)
for i, (p_val, g) in enumerate(zip(ps, gaps)):
    ax1.annotate(f"p={p_val}", (p_val, g), textcoords="offset points",
                 xytext=(5, 5), fontsize=8)

ax2.scatter(ps, accs, c=cols, s=120, zorder=5, edgecolors="white", linewidths=1.5)
ax2.plot(ps, accs, "--", color="gray", linewidth=1, zorder=1)
ax2.set_xlabel("Dropout Rate p"); ax2.set_ylabel("Validation Accuracy (%)")
ax2.set_title("Val Accuracy vs Dropout (pick the peak)")
ax2.grid(True, alpha=0.3)
best_p_val = ps[best_idx]
ax2.axvline(best_p_val, color="gold", linestyle="--", linewidth=2, label=f"Best p={best_p_val}")
ax2.legend()
for i, (p_val, a) in enumerate(zip(ps, accs)):
    ax2.annotate(f"{a:.1f}%", (p_val, a), textcoords="offset points",
                 xytext=(5, 5), fontsize=8)

plt.tight_layout()
p4 = os.path.join(OUT_DIR, "04_gap_analysis.png")
plt.savefig(p4, dpi=120, bbox_inches="tight"); plt.close()
print(f"    Saved: {p4}")

# ── Plot 5: Feature distribution before/after BN ─────────────────────────────
print("    Computing feature distributions …")

torch.manual_seed(0)
raw_x = torch.randn(256, 64) * 5 + 3   # deliberately shifted + scaled

bn  = nn.BatchNorm1d(64)
ln  = nn.LayerNorm(64)
bn.eval(); ln.eval()
with torch.no_grad():
    out_bn = bn(raw_x)
    out_ln = ln(raw_x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Feature Distribution: Before vs After Normalization",
             fontsize=13, fontweight="bold")

axes[0].hist(raw_x[:, 0].numpy(), bins=40, color="#e74c3c", alpha=0.7, edgecolor="white")
axes[0].set_title(f"RAW  (μ={raw_x[:,0].mean():.1f}, σ={raw_x[:,0].std():.1f})")
axes[0].set_xlabel("Activation value"); axes[0].set_ylabel("Count")
axes[0].axvline(0, color="black", linestyle="--")

axes[1].hist(out_bn[:, 0].detach().numpy(), bins=40, color="#3498db", alpha=0.7, edgecolor="white")
axes[1].set_title(f"BatchNorm  (μ≈{out_bn[:,0].mean():.2f}, σ≈{out_bn[:,0].std():.2f})")
axes[1].set_xlabel("Activation value"); axes[1].axvline(0, color="black", linestyle="--")

axes[2].hist(out_ln[:, 0].detach().numpy(), bins=40, color="#2ecc71", alpha=0.7, edgecolor="white")
axes[2].set_title(f"LayerNorm  (μ≈{out_ln[:,0].mean():.2f}, σ≈{out_ln[:,0].std():.2f})")
axes[2].set_xlabel("Activation value"); axes[2].axvline(0, color="black", linestyle="--")

plt.tight_layout()
p5 = os.path.join(OUT_DIR, "05_feature_distributions.png")
plt.savefig(p5, dpi=120, bbox_inches="tight"); plt.close()
print(f"    Saved: {p5}")

# ── Plot 6: Normalization cheat sheet (visual summary) ─────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis("off")
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

table_data = [
    ["Method",          "Normalise Over",   "Batch Dep?", "Sequence Dep?", "Best Use Case"],
    ["BatchNorm",       "Batch × Spatial",  "YES ✗",      "No",            "CNNs, large batches"],
    ["LayerNorm",       "Features",         "No ✓",       "No ✓",          "Transformers, RNNs, NLP"],
    ["InstanceNorm",    "Spatial (per ch)", "No ✓",       "No",            "Style transfer, GAN"],
    ["GroupNorm",       "Group of channels","No ✓",       "No",            "Detection, small batch"],
    ["SpectralNorm",    "Weight matrix",    "No ✓",       "No",            "GANs, stability"],
]
colors_table = [
    ["#1e2a38"]*5,
    ["#1a1a2e","#16213e","#1a1a2e","#16213e","#1a1a2e"],
    ["#1a1a2e","#16213e","#1a1a2e","#16213e","#1a1a2e"],
    ["#1a1a2e","#16213e","#1a1a2e","#16213e","#1a1a2e"],
    ["#1a1a2e","#16213e","#1a1a2e","#16213e","#1a1a2e"],
    ["#1a1a2e","#16213e","#1a1a2e","#16213e","#1a1a2e"],
]

tbl = ax.table(cellText=table_data, loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#2d3748")
    cell.set_linewidth(0.5)
    if r == 0:
        cell.set_facecolor("#2b3a55")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        cell.set_facecolor("#161b27")
        tc = ["#e2e8f0","#63b3ed","#fc8181" if "YES" in table_data[r][c] else "#68d391","#a0aec0","#fbd38d"]
        cell.set_text_props(color=tc[c] if c < len(tc) else "white")

ax.set_title("Normalization Methods — Quick Reference", color="white",
             fontsize=14, fontweight="bold", pad=20)

plt.tight_layout()
p6 = os.path.join(OUT_DIR, "06_norm_cheat_sheet.png")
plt.savefig(p6, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"    Saved: {p6}")


# ─────────────────────────────────────────────────────────────────────────────
# 11.  FINAL DECISION GUIDE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("  FINAL DECISION GUIDE")
print("="*72)

guide = f"""
  NORMALIZATION — WHICH TO USE?
  ──────────────────────────────
  ┌──────────────────────────────────────────────────────────────────┐
  │  Situation                         │  Use                        │
  ├──────────────────────────────────────────────────────────────────┤
  │  CNN + batch ≥ 16                  │  BatchNorm2d  ← default     │
  │  Transformer / BERT / GPT          │  LayerNorm    ← standard    │
  │  RNN / LSTM / GRU                  │  LayerNorm                  │
  │  Style transfer / image-to-image   │  InstanceNorm               │
  │  Object detection / small batch    │  GroupNorm (G=32)           │
  │  GAN training (discriminator)      │  SpectralNorm               │
  │  batch = 1 (online / RL)           │  LayerNorm or GroupNorm     │
  └──────────────────────────────────────────────────────────────────┘

  DROPOUT — HOW TO CHOOSE p?
  ───────────────────────────
  Ablation results on this run:

  Best dropout rate found : {best_p_val}
  Best validation accuracy: {accs[best_idx]:.2f}%

  ┌────────────────────────────────────────────────────────────────┐
  │  Signal                    │  Action                           │
  ├────────────────────────────────────────────────────────────────┤
  │  train_acc >> val_acc      │  INCREASE p  (overfitting)        │
  │  train_acc ≈ val_acc       │  p is about right                 │
  │  both accuracies low       │  DECREASE p  (underfitting)       │
  │  Small dataset (< 5K)      │  Start at p = 0.5                 │
  │  Large dataset (> 100K)    │  Start at p = 0.1–0.2             │
  │  After BatchNorm           │  p = 0.2–0.3  (BN already regs)  │
  │  FC layers                 │  p = 0.4–0.5                      │
  │  Conv layers               │  p = 0.1–0.2  (use Dropout2d)     │
  └────────────────────────────────────────────────────────────────┘

  PRACTICAL RECIPE (copy-paste template):
  ────────────────────────────────────────
    # Convolutional block (after each conv stage)
    nn.Conv2d(64, 128, 3, padding=1)
    nn.BatchNorm2d(128)              # ← always after conv, before activation
    nn.ReLU()
    nn.Dropout2d(p=0.2)             # ← low p, drops channels

    # Fully connected block
    nn.Linear(512, 256)
    nn.LayerNorm(256)  # or BatchNorm1d(256)
    nn.ReLU()
    nn.Dropout(p=0.4)               # ← higher p on dense layers

    nn.Linear(256, num_classes)     # ← NO dropout on output layer!
"""
print(guide)

print("="*72)
print(f"  All plots saved to: ./{OUT_DIR}/")
for p in [p1, p2, p3, p4, p5, p6]:
    print(f"    • {p}")
print("="*72)
print("  Done! 🎉")
print("="*72)


  [5] EXPERIMENT A: Normalization Types (dropout fixed at 0.3)

  ── No Norm + No Dropout ──
     Params: 698,154
    No Norm + No  Ep  1/8 | Train 22.3% loss 2.063 | Val 32.2% loss 1.865 | gap -9.9%
    No Norm + No  Ep  2/8 | Train 30.9% loss 1.826 | Val 36.5% loss 1.734 | gap -5.6%
    No Norm + No  Ep  3/8 | Train 35.7% loss 1.690 | Val 39.6% loss 1.706 | gap -3.9%
    No Norm + No  Ep  4/8 | Train 39.2% loss 1.627 | Val 42.1% loss 1.559 | gap -2.9%
    No Norm + No  Ep  5/8 | Train 42.8% loss 1.533 | Val 40.6% loss 1.593 | gap 2.2%
    No Norm + No  Ep  6/8 | Train 45.3% loss 1.464 | Val 49.0% loss 1.418 | gap -3.7%
    No Norm + No  Ep  7/8 | Train 47.9% loss 1.397 | Val 50.7% loss 1.361 | gap -2.8%
    No Norm + No  Ep  8/8 | Train 49.4% loss 1.374 | Val 52.3% loss 1.321 | gap -3.0%

  ── Batch Norm ──
     Params: 698,986
    Batch Norm  Ep  1/8 | Train 30.5% loss 1.871 | Val 43.2% loss 1.561 | gap -12.8%
    Batch Norm  Ep  2/8 | Train 42.0% loss 1.576 | Val 48.1% loss 1.378 